# Stage 3 — Out-of-Sample (OOS-1) Validation

Per [METHODOLOGY.md](../METHODOLOGY.md) Stage 3:

1. Run each of the **top 10 IS candidates** on the OOS-1 partition.
2. **Acceptance gate:** OOS-1 PF must be ≥ 70% of IS PF.
3. Select **one** candidate by OOS-1 Sharpe (no ensembling).

If the gate fails, the strategy is declared overfit and the cycle terminates with a postmortem.

In [ ]:
from __future__ import annotations

import json
import pickle
from dataclasses import asdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from quant.backtest.costs import CostMode, get_cost_model
from quant.backtest.metrics import compute_metrics
from quant.strategies.mean_reversion import MeanReversionConfig, MeanReversionStrategy

DATA_PROC = Path('../data/processed')
RESULTS = Path('../results')
PARAM_KEYS = ['min_range_pct', 'extreme_proximity_pct', 'stop_buffer_pct', 'tp_fraction', 'time_stop_hours']
ACCEPTANCE_RATIO = 0.70

## 1. Load split + top-10 candidates

In [ ]:
with open(DATA_PROC / 'split_BTCUSDT_1h.pkl', 'rb') as f:
    split = pickle.load(f)
candidates = json.loads((DATA_PROC / 'top10_is_candidates.json').read_text())

print(f'OOS-1 bars: {len(split.oos1_data):,}  [{split.oos1_range[0]}  →  {split.oos1_range[1]}]')
print(f'Candidates: {len(candidates)}')

## 2. Evaluate each candidate on OOS-1

In [ ]:
base = MeanReversionConfig(name='mean_reversion', timeframe='1h')
cost = get_cost_model(CostMode.HORROR)

rows = []
for cand in candidates:
    params = {k: cand[k] for k in PARAM_KEYS}
    cfg = MeanReversionConfig(**{**asdict(base), **params})
    strat = MeanReversionStrategy(cfg)
    trades = strat.simulate(split.oos1_data, cost_model=cost)
    metrics = compute_metrics(trades)
    rows.append({
        **params,
        'is_sharpe': cand['is_sharpe'],
        'is_pf': cand['is_pf'],
        'oos_n_trades': metrics['n_trades'],
        'oos_win_rate': metrics['win_rate'],
        'oos_pf': metrics['pf'],
        'oos_sharpe': metrics['sharpe'],
        'oos_max_dd': metrics['max_dd'],
        'oos_cagr': metrics['cagr'],
    })

oos = pd.DataFrame(rows)
oos['pf_retention'] = (oos['oos_pf'] / oos['is_pf']).round(3)
oos['passes_gate'] = oos['pf_retention'] >= ACCEPTANCE_RATIO
oos.round(4)

## 3. Acceptance gate evaluation

In [ ]:
n_passing = int(oos['passes_gate'].sum())
print(f'Candidates passing OOS PF ≥ {ACCEPTANCE_RATIO * 100:.0f}% of IS PF: {n_passing} / {len(oos)}')
if n_passing == 0:
    print('\n⚠️  No candidate passes the OOS retention gate.')
    print('Per METHODOLOGY.md Stage 3: strategy declared OVERFIT.')
    print('Action: record in results/POSTMORTEM.md and terminate cycle.')
else:
    print('\n✓ At least one candidate passes — proceed to selection.')

## 4. Select chosen configuration

Among gate-passing candidates, choose **one** by highest OOS Sharpe. No ensembling, no parameter averaging.

In [ ]:
passing = oos[oos['passes_gate']].copy()
if len(passing) > 0:
    chosen_row = passing.sort_values('oos_sharpe', ascending=False).iloc[0]
    chosen_params = {k: (int(chosen_row[k]) if k == 'time_stop_hours' else float(chosen_row[k])) for k in PARAM_KEYS}
    chosen_metadata = {
        'params': chosen_params,
        'is_sharpe': float(chosen_row['is_sharpe']),
        'is_pf': float(chosen_row['is_pf']),
        'oos_sharpe': float(chosen_row['oos_sharpe']),
        'oos_pf': float(chosen_row['oos_pf']),
        'oos_max_dd': float(chosen_row['oos_max_dd']),
        'pf_retention': float(chosen_row['pf_retention']),
        'oos_n_trades': int(chosen_row['oos_n_trades']),
    }
    chosen_path = DATA_PROC / 'chosen_config.json'
    chosen_path.write_text(json.dumps(chosen_metadata, indent=2))
    print('Chosen configuration:')
    print(json.dumps(chosen_metadata, indent=2))
    print(f'\nWrote: {chosen_path}')
else:
    print('No chosen config — strategy retired at Stage 3.')

## 5. IS vs OOS comparison plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].bar(np.arange(len(oos)) - 0.2, oos['is_sharpe'], 0.4, label='IS Sharpe', color='#3b82f6')
axes[0].bar(np.arange(len(oos)) + 0.2, oos['oos_sharpe'], 0.4, label='OOS Sharpe', color='#10b981')
axes[0].axhline(0, color='black', lw=0.5)
axes[0].set_xlabel('Candidate rank (by IS Sharpe)')
axes[0].set_ylabel('Sharpe')
axes[0].set_title('IS vs OOS Sharpe — top 10')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].scatter(oos['is_pf'], oos['oos_pf'], s=80, c=oos['passes_gate'].map({True: '#10b981', False: '#ef4444'}), alpha=0.8)
axes[1].plot([0, oos['is_pf'].max() * 1.1], [0, oos['is_pf'].max() * 1.1 * ACCEPTANCE_RATIO], 'k--', alpha=0.4, label=f'70% retention')
axes[1].axhline(1.0, color='gray', lw=0.5)
axes[1].axvline(1.0, color='gray', lw=0.5)
axes[1].set_xlabel('IS PF')
axes[1].set_ylabel('OOS PF')
axes[1].set_title('PF retention scatter')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
out = RESULTS / 'figures' / '04_is_vs_oos.png'
out.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(out, dpi=120, bbox_inches='tight')
print(f'Wrote: {out}')
plt.show()

## Stage 3 complete

- All 10 IS candidates evaluated on OOS-1 with HORROR costs.
- Acceptance gate applied: OOS PF ≥ 70% of IS PF.
- Chosen configuration (if any) persisted to `data/processed/chosen_config.json`.

Proceed to:
- [05_walkforward.ipynb](05_walkforward.ipynb) — Stage 4 regime robustness on the chosen config.
- [06_holdout.ipynb](06_holdout.ipynb) — Stage 5 single-shot final test.

If no candidate passed, jump to [08_postmortem.ipynb](08_postmortem.ipynb) and write the entry.